# 02 - PySpark SDN DDoS Dataset Analytics

This notebook loads the generated CSV with PySpark, cleans the data, applies numerical and categorical preprocessing, and creates five Plotly visualizations. Each chart includes a short purpose and interpretation.

The dataset is a controlled local-lab profile. It is suitable for exploratory analysis and pipeline testing, not claims about live Internet traffic.

In [1]:
# Install packages once if necessary.
# Spark itself should be installed in the WSL environment.
#
# %pip install -q pyspark plotly pandas pyarrow

import sys, os
print(sys.version)


3.10.12 (main, Jun 22 2026, 18:55:27) [GCC 11.4.0]


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, LongType, StringType, TimestampType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

spark = (
    SparkSession.builder
    .appName("SDN-DDoS-Analytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/23 23:33:59 WARN Utils: Your hostname, SATYASPALADUGU, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/23 23:33:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/23 23:34:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.2.0


In [3]:
from pathlib import Path

DATA_PATH = Path.cwd() / "data" / "processed" / "sdn_ddos_controlled_lab.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}. Run Network and Traffic/scripts/run_experiment.sh first.")

df = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_PATH))

print("Rows:", df.count())
print("Columns:", len(df.columns))
df.printSchema()
df.show(5, truncate=False)


Rows: 110000
Columns: 28
root
 |-- timestamp_utc: timestamp (nullable = true)
 |-- experiment_id: string (nullable = true)
 |-- scenario_id: string (nullable = true)
 |-- switch_id: string (nullable = true)
 |-- src_host: string (nullable = true)
 |-- dst_host: string (nullable = true)
 |-- protocol: string (nullable = true)
 |-- application: string (nullable = true)
 |-- window_seconds: double (nullable = true)
 |-- packet_count: integer (nullable = true)
 |-- byte_count: integer (nullable = true)
 |-- flow_count: integer (nullable = true)
 |-- packet_rate: double (nullable = true)
 |-- byte_rate: double (nullable = true)
 |-- flow_rate: double (nullable = true)
 |-- mean_packet_size: double (nullable = true)
 |-- tcp_syn_count: integer (nullable = true)
 |-- tcp_ack_count: integer (nullable = true)
 |-- tcp_handshake_completion_ratio: double (nullable = true)
 |-- flow_table_size: integer (nullable = true)
 |-- packet_in_count: integer (nullable = true)
 |-- controller_cpu_proxy_pct:

26/08/23 23:34:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------------+-------------------------+----------------+---------+--------+--------+--------+-----------+--------------+------------+----------+----------+-----------+---------+---------+-----------------+-------------+-------------+------------------------------+---------------+---------------+------------------------+----------------+--------------+------------------------+------+-------------+----------------------+
|timestamp_utc             |experiment_id            |scenario_id     |switch_id|src_host|dst_host|protocol|application|window_seconds|packet_count|byte_count|flow_count|packet_rate|byte_rate|flow_rate|mean_packet_size |tcp_syn_count|tcp_ack_count|tcp_handshake_completion_ratio|flow_table_size|packet_in_count|controller_cpu_proxy_pct|attack_intensity|attacker_count|background_traffic_level|label |attack_family|data_source           |
+--------------------------+-------------------------+----------------+---------+--------+--------+--------+-----------+----

## 1. Data cleaning

Cleaning steps:

- Remove exact duplicate rows.
- Cast numeric fields to appropriate numeric types.
- Parse timestamps.
- Replace invalid negative measurements with null.
- Remove records missing essential identifiers/labels.
- Check consistency of packet/byte rates.

We keep the cleaning logic explicit so it can be documented in the project report.


In [4]:
raw_count = df.count()

clean = df.dropDuplicates()

numeric_cols = [
    "window_seconds", "packet_count", "byte_count", "flow_count",
    "packet_rate", "byte_rate", "flow_rate", "mean_packet_size",
    "tcp_syn_count", "tcp_ack_count", "tcp_handshake_completion_ratio",
    "flow_table_size", "packet_in_count", "controller_cpu_proxy_pct",
    "attacker_count", "background_traffic_level"
]

for col in numeric_cols:
    clean = clean.withColumn(col, F.col(col).cast("double"))

clean = clean.dropna(subset=["scenario_id", "timestamp_utc", "protocol", "label"])
clean = clean.filter(
    (F.col("packet_count") >= 0) &
    (F.col("byte_count") >= 0) &
    (F.col("packet_rate") >= 0) &
    F.col("tcp_handshake_completion_ratio").between(0, 1)
)

clean_count = clean.count()

print("Raw rows:", raw_count)
print("Clean rows:", clean_count)
print("Removed:", raw_count - clean_count)


Raw rows: 110000
Clean rows: 110000
Removed: 0


In [5]:
# Missing-value profile
missing = clean.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in clean.columns
])

missing.show(truncate=False)


+-------------+-------------+-----------+---------+--------+--------+--------+-----------+--------------+------------+----------+----------+-----------+---------+---------+----------------+-------------+-------------+------------------------------+---------------+---------------+------------------------+----------------+--------------+------------------------+-----+-------------+-----------+
|timestamp_utc|experiment_id|scenario_id|switch_id|src_host|dst_host|protocol|application|window_seconds|packet_count|byte_count|flow_count|packet_rate|byte_rate|flow_rate|mean_packet_size|tcp_syn_count|tcp_ack_count|tcp_handshake_completion_ratio|flow_table_size|packet_in_count|controller_cpu_proxy_pct|attack_intensity|attacker_count|background_traffic_level|label|attack_family|data_source|
+-------------+-------------+-----------+---------+--------+--------+--------+-----------+--------------+------------+----------+----------+-----------+---------+---------+----------------+-------------+-------

In [6]:
# Exploratory data analysis
# Spark summaries inspect data quality, distributions, class composition,
# and relationships between the main traffic features.

In [7]:
# Dataset shape, time span, class balance, protocol composition, and feature statistics.
eda_overview = clean.select(
    F.count("*").alias("rows"),
    F.countDistinct("label").alias("classes"),
    F.countDistinct("protocol").alias("protocols"),
    F.min("timestamp_utc").alias("first_timestamp"),
    F.max("timestamp_utc").alias("last_timestamp")
)
eda_overview.show(truncate=False)

print("Schema after cleaning:")
clean.printSchema()

summary_cols = [
    "packet_rate", "byte_rate", "flow_rate", "mean_packet_size",
    "tcp_handshake_completion_ratio", "flow_table_size",
    "packet_in_count", "controller_cpu_proxy_pct"
]
clean.select(summary_cols).summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).show()

class_summary = (
    clean.groupBy("label", "attack_family", "protocol")
    .agg(
        F.count("*").alias("records"),
        F.round(100 * F.count("*") / clean_count, 2).alias("share_pct"),
        F.round(F.avg("packet_rate"), 2).alias("mean_packet_rate"),
        F.round(F.avg("byte_rate"), 2).alias("mean_byte_rate"),
        F.round(F.avg("controller_cpu_proxy_pct"), 2).alias("mean_controller_pressure")
    )
    .orderBy(F.desc("records"), "label")
)
class_summary.show(20, truncate=False)

protocol_summary = (
    clean.groupBy("protocol", "label")
    .count()
    .orderBy("protocol", "label")
)
protocol_summary.show(30, truncate=False)

missing_profile = clean.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in clean.columns
])
missing_profile.show(truncate=False)

correlation_cols = [
    "packet_rate", "byte_rate", "flow_rate", "mean_packet_size",
    "flow_table_size", "packet_in_count", "controller_cpu_proxy_pct"
]
correlations = {
    f"{left}__{right}": round(clean.stat.corr(left, right), 3)
    for index, left in enumerate(correlation_cols)
    for right in correlation_cols[index + 1:]
}
print("Selected numeric feature correlations:")
for pair, value in correlations.items():
    print(f"{pair}: {value}")

+------+-------+---------+--------------------------+--------------------------+
|rows  |classes|protocols|first_timestamp           |last_timestamp            |
+------+-------+---------+--------------------------+--------------------------+
|110000|11     |3        |2026-08-23 23:06:53.469483|2026-08-23 23:06:57.383905|
+------+-------+---------+--------------------------+--------------------------+

Schema after cleaning:
root
 |-- timestamp_utc: timestamp (nullable = true)
 |-- experiment_id: string (nullable = true)
 |-- scenario_id: string (nullable = true)
 |-- switch_id: string (nullable = true)
 |-- src_host: string (nullable = true)
 |-- dst_host: string (nullable = true)
 |-- protocol: string (nullable = true)
 |-- application: string (nullable = true)
 |-- window_seconds: double (nullable = true)
 |-- packet_count: double (nullable = true)
 |-- byte_count: double (nullable = true)
 |-- flow_count: double (nullable = true)
 |-- packet_rate: double (nullable = true)
 |-- byte

+-------+-----------------+-----------------+------------------+------------------+------------------------------+------------------+------------------+------------------------+
|summary|      packet_rate|        byte_rate|         flow_rate|  mean_packet_size|tcp_handshake_completion_ratio|   flow_table_size|   packet_in_count|controller_cpu_proxy_pct|
+-------+-----------------+-----------------+------------------+------------------+------------------------------+------------------+------------------+------------------------+
|  count|           110000|           110000|            110000|            110000|                        110000|            110000|            110000|                  110000|
|   mean|957.0215918181837|772549.1271000009|11.325118181818178|1029.1805017848214|           0.17339046296022081|225.88554545454545|118.33682727272728|      45.404692205057394|
| stddev|717.6674336273072| 819969.927751038|14.733766623996294|1525.5385380579462|             0.279071775122

+-----------------------+-------------+--------+-------+---------+----------------+--------------+------------------------+
|label                  |attack_family|protocol|records|share_pct|mean_packet_rate|mean_byte_rate|mean_controller_pressure|
+-----------------------+-------------+--------+-------+---------+----------------+--------------+------------------------+
|BENIGN                 |BENIGN       |TCP     |10000  |9.09     |99.58           |79965.47      |14.94                   |
|HTTP_FLOOD             |APPLICATION  |TCP     |10000  |9.09     |899.36          |763092.64     |47.46                   |
|ICMP_FLOOD             |VOLUMETRIC   |ICMP    |10000  |9.09     |1405.06         |892882.58     |40.04                   |
|LAND_STYLE             |PROTOCOL     |TCP     |10000  |9.09     |449.99          |188426.27     |42.51                   |
|PACKET_IN_FLOOD_STYLE  |SDN_CONTROL  |UDP     |10000  |9.09     |1094.04         |361664.38     |77.39                   |
|PING_OF

+--------+-----------------------+-----+
|protocol|label                  |count|
+--------+-----------------------+-----+
|ICMP    |ICMP_FLOOD             |10000|
|ICMP    |PING_OF_DEATH_STYLE    |10000|
|ICMP    |SMURF_STYLE            |10000|
|TCP     |BENIGN                 |10000|
|TCP     |HTTP_FLOOD             |10000|
|TCP     |LAND_STYLE             |10000|
|TCP     |SLOWLORIS_STYLE        |10000|
|TCP     |SYN_FLOOD              |10000|
|UDP     |PACKET_IN_FLOOD_STYLE  |10000|
|UDP     |UDP_AMPLIFICATION_STYLE|10000|
|UDP     |UDP_FLOOD              |10000|
+--------+-----------------------+-----+



+-------------+-------------+-----------+---------+--------+--------+--------+-----------+--------------+------------+----------+----------+-----------+---------+---------+----------------+-------------+-------------+------------------------------+---------------+---------------+------------------------+----------------+--------------+------------------------+-----+-------------+-----------+
|timestamp_utc|experiment_id|scenario_id|switch_id|src_host|dst_host|protocol|application|window_seconds|packet_count|byte_count|flow_count|packet_rate|byte_rate|flow_rate|mean_packet_size|tcp_syn_count|tcp_ack_count|tcp_handshake_completion_ratio|flow_table_size|packet_in_count|controller_cpu_proxy_pct|attack_intensity|attacker_count|background_traffic_level|label|attack_family|data_source|
+-------------+-------------+-----------+---------+--------+--------+--------+-----------+--------------+------------+----------+----------+-----------+---------+---------+----------------+-------------+-------

Selected numeric feature correlations:
packet_rate__byte_rate: 0.572
packet_rate__flow_rate: 0.071
packet_rate__mean_packet_size: -0.194
packet_rate__flow_table_size: 0.067
packet_rate__packet_in_count: 0.061
packet_rate__controller_cpu_proxy_pct: 0.336
byte_rate__flow_rate: -0.247
byte_rate__mean_packet_size: 0.506
byte_rate__flow_table_size: -0.231
byte_rate__packet_in_count: -0.206
byte_rate__controller_cpu_proxy_pct: 0.028
flow_rate__mean_packet_size: -0.27
flow_rate__flow_table_size: 0.939
flow_rate__packet_in_count: 0.837
flow_rate__controller_cpu_proxy_pct: 0.59
mean_packet_size__flow_table_size: -0.253
mean_packet_size__packet_in_count: -0.225
mean_packet_size__controller_cpu_proxy_pct: -0.234
flow_table_size__packet_in_count: 0.786
flow_table_size__controller_cpu_proxy_pct: 0.554
packet_in_count__controller_cpu_proxy_pct: 0.492


## 2. Preprocessing

We separate:

### Numerical features
Scaled with `StandardScaler` after vector assembly.

### Categorical features
`protocol` and `attack_intensity` are converted using `StringIndexer` + `OneHotEncoder`.

### Identifier columns
IPs and experiment identifiers are retained for analysis but are **not automatically treated as ML features**, because raw addresses can cause severe leakage and memorization.

The same principle applies to `scenario_id`: it is metadata, not a predictive feature.


In [8]:
numeric_features = [
    "packet_rate",
    "byte_rate",
    "flow_rate",
    "mean_packet_size",
    "tcp_syn_count",
    "tcp_ack_count",
    "tcp_handshake_completion_ratio",
    "flow_table_size",
    "packet_in_count",
    "controller_cpu_proxy_pct",
    "attacker_count",
    "background_traffic_level",
]

categorical_features = ["protocol", "attack_intensity"]

indexers = [
    StringIndexer(inputCol=x, outputCol=f"{x}_idx", handleInvalid="keep")
    for x in categorical_features
]

encoders = [
    OneHotEncoder(inputCol=f"{x}_idx", outputCol=f"{x}_ohe")
    for x in categorical_features
]

assembler = VectorAssembler(
    inputCols=numeric_features + [f"{x}_ohe" for x in categorical_features],
    outputCol="features_raw",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features_scaled",
    withStd=True,
    withMean=False
)

preprocess_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])

processed = preprocess_pipeline.fit(clean).transform(clean)

processed.select(
    "label", "protocol", "features_scaled"
).show(5, truncate=False)


+------+--------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label |protocol|features_scaled                                                                                                                                                                                                                                                                         |
+------+--------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|BENIGN|TCP     |[0.20831375787024636,0.13694380269283948,0.04750991500435049,0.4923522562027823,0.2503

## Visualization 1 - Class distribution

Purpose: check whether the dataset contains all traffic classes and whether the classes are balanced.

Interpretation: equal counts indicate a balanced development dataset; production traffic would usually be much more benign-heavy.

In [9]:
import plotly.express as px
import pandas as pd

class_pd = (
    clean.groupBy("label")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

fig1 = px.bar(
    class_pd,
    x="label",
    y="count",
    color="label",
    title="SDN DDoS Dataset - Class Distribution",
    labels={"label":"Traffic class", "count":"Observations"}
)
fig1.update_layout(showlegend=False, xaxis_tickangle=-45)
fig1.show()


## Visualization 2 - Packet-rate behavior

Purpose: compare traffic intensity across labels using packets per second.

Interpretation: high-rate flood profiles should separate from benign traffic, while low-rate attacks demonstrate why packet rate alone is insufficient.

In [10]:
packet_pd = (
    clean
    .select("label", "attack_family", "packet_rate")
    .sample(False, 0.10, seed=42)
    .limit(12000)
    .toPandas()
)

fig2 = px.box(
    packet_pd,
    x="label",
    y="packet_rate",
    color="attack_family",
    points=False,
    title="Packet-Rate Distribution by Traffic Class",
    labels={
        "label": "Traffic class",
        "attack_family": "Attack family",
        "packet_rate": "Packets/second"
    }
)
fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

## Visualization 3 - Bubble chart: traffic intensity and flow-table pressure

Purpose: compare packet rate and byte rate while using bubble size for flow-table size and color for traffic class.

Interpretation: the chart exposes traffic classes that combine high volume with high flow-table pressure, a useful SDN-specific view of attack behavior.

In [11]:
bubble_pd = (
    clean
    .select(
        "label", "attack_family", "packet_rate", "byte_rate",
        "flow_table_size", "controller_cpu_proxy_pct"
    )
    .sample(False, 0.25, seed=42)
    .limit(12000)
    .toPandas()
)

fig3 = px.scatter(
    bubble_pd,
    x="packet_rate",
    y="byte_rate",
    size="flow_table_size",
    color="label",
    hover_data=["attack_family", "controller_cpu_proxy_pct"],
    size_max=32,
    opacity=0.65,
    title="Bubble Chart: Traffic Intensity and Flow-Table Size",
    labels={
        "packet_rate": "Packets/second",
        "byte_rate": "Bytes/second",
        "flow_table_size": "Flow-table entries",
        "label": "Traffic class"
    }
)
fig3.show()

## Visualization 4 - TCP handshake completion

Purpose: examine whether TCP connection attempts complete successfully across TCP traffic classes.

Interpretation: SYN-flood and slow-rate profiles should show lower completion than normal TCP; non-TCP records are excluded.

In [12]:
tcp_pd = (
    clean
    .filter(F.col("protocol") == "TCP")
    .groupBy("label")
    .agg(F.avg("tcp_handshake_completion_ratio").alias("completion_ratio"))
    .orderBy("completion_ratio")
    .toPandas()
)

fig4 = px.bar(
    tcp_pd,
    x="label",
    y="completion_ratio",
    color="label",
    title="TCP Connection Completion Ratio",
    labels={"label":"Traffic class", "completion_ratio":"Completion ratio"}
)
fig4.update_layout(showlegend=False, xaxis_tickangle=-45)
fig4.show()


## Visualization 5 - SDN control-plane pressure

Purpose: compare controller CPU proxy and Packet-In activity across traffic classes.

Interpretation: Packet-In flood should create the strongest control-plane signal, showing why SDN telemetry complements ordinary packet statistics.

In [13]:
control_pd = (
    clean.groupBy("label")
    .agg(
        F.avg("controller_cpu_proxy_pct").alias("mean_controller_load"),
        F.avg("packet_in_count").alias("mean_packet_in")
    )
    .orderBy(F.desc("mean_controller_load"))
    .toPandas()
)

fig5 = px.bar(
    control_pd,
    x="label",
    y="mean_controller_load",
    color="label",
    title="Mean SDN Controller-Load Indicator by Traffic Class",
    labels={"label":"Traffic class", "mean_controller_load":"Controller-load indicator"}
)
fig5.update_layout(showlegend=False, xaxis_tickangle=-45)
fig5.show()


# Analytical summary

The EDA combines dataset quality, class balance, protocol composition, numerical distributions, and feature relationships. The five visualizations then focus on:

1. Class balance.
2. Packet-rate distribution by traffic class.
3. Packet rate, byte rate, and flow-table pressure in a bubble chart.
4. TCP handshake completion behavior.
5. SDN controller pressure and Packet-In activity.

Together, these views connect the dataset's network-volume, protocol, flow, and SDN control-plane signals. The scaled Spark feature vector is ready for downstream model development, while the raw identifiers and labels remain excluded from preprocessing features to reduce leakage.